# 05 (bonus) - YOLO, and the two algorithms inside it

Assignments 01-04 are *image processing*: one pixel at a time, no notion of objects.
This notebook is the jump to *computer vision*: getting a box and a label out of an image.

You will not train anything here. You will run a pretrained detector, and then implement
the two pieces of plain arithmetic every detector depends on - **IoU** and **non-maximum
suppression**. Those two are also the most commonly asked detection questions in a viva,
and they are 5 lines each, so there is no excuse not to be able to write them.

### Install (once)
Ultralytics is not installed yet. Run the next cell once; it pulls ~50 MB of package
(torch is already present) plus a 6 MB weights file on first use.

In [ ]:
import torch
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
# already installed in the cv env -- this is here only if you rebuild the env
# %pip install -q ultralytics

In [ ]:
import cv2
import matplotlib.pyplot as plt
# the graders are the test harness -- everything else is written out in full
from cvlab import check_iou, check_nms

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE)

## Run the detector

`yolov8n` is the nano model: 3.2M parameters, ~8.7 GFLOPs at 640x640, trained on COCO's
80 classes. The `n` is the right choice on an 8 GB machine - see the sizing table at the bottom.

Ultralytics wants BGR (it wraps OpenCV), so pass a path or a BGR array, not our RGB one.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")          # downloads once, then cached in ~/.cache or cwd
res = model("images/messi5.jpg", device=DEVICE, verbose=False)[0]

print("boxes found:", len(res.boxes))
for b in res.boxes:
    cls  = model.names[int(b.cls)]
    conf = float(b.conf)
    x1, y1, x2, y2 = b.xyxy[0].tolist()
    print(f"  {cls:12} {conf:.2f}  ({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})")

## Task 1 - draw the boxes yourself

`res.plot()` would do this for you. Don't use it. Drawing boxes with `cv2.rectangle` and
`cv2.putText` is a skill you will use in every CV project, and it forces you to understand
the coordinate convention: `xyxy` is `(left, top, right, bottom)` in pixels, y growing
**downward**, origin top-left.

No automated grader here - the check is whether the picture looks right.

In [ ]:
def my_draw_boxes(rgb, boxes, labels, scores, color=(0, 255, 0)):
    """Draw labelled bounding boxes on a copy of an RGB image.

    Parameters
    ----------
    rgb    : (H, W, 3) uint8 RGB array
    boxes  : list of (x1, y1, x2, y2) in pixels
    labels : list of class-name strings, same length as boxes
    scores : list of floats 0..1, same length as boxes

    Returns
    -------
    a new (H, W, 3) uint8 array with a rectangle per box and "label 0.87" text above it.
    Must not modify the input array.
    """
    # TODO: copy the image, then loop: cv2.rectangle(...) and cv2.putText(...)
    raise NotImplementedError

In [ ]:
import cv2
rgb    = cv2.cvtColor(cv2.imread("images/messi5.jpg"), cv2.COLOR_BGR2RGB)
boxes  = [tuple(b.xyxy[0].tolist()) for b in res.boxes]
labels = [model.names[int(b.cls)] for b in res.boxes]
scores = [float(b.conf) for b in res.boxes]

out = my_draw_boxes(rgb, boxes, labels, scores)
assert out is not rgb and not np.array_equal(out, rgb), "you either mutated the input or drew nothing"
_imgs, _titles = [rgb, out], ["input", "your drawing"]
fig, axes = plt.subplots(1, 2, figsize=(6 * 2, 6), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

`cv2.rectangle(img, (x1,y1), (x2,y2), color, thickness)` needs **int** corners - the model gives you floats, so cast.

`cv2.putText(img, text, (x,y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)` - `(x,y)` is the text *baseline*, so `y1 - 5` puts the label just above the box, and clamp it so a box at the very top does not draw off-screen.

`img.copy()` is how you avoid mutating the caller's array.

</details>

## Task 2 - IoU

Intersection over Union. Every detection metric, every NMS implementation, and every
assignment-of-prediction-to-ground-truth step is built on this one number.

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

The trap is the intersection of two boxes that do not overlap: the naive formula gives a
*negative* width, whose product with a negative height is a positive area. Clamp at zero.

In [ ]:
def my_iou(box_a, box_b):
    """Intersection-over-Union of two axis-aligned boxes.

    Parameters
    ----------
    box_a, box_b : (x1, y1, x2, y2) with x1 < x2 and y1 < y2

    Returns
    -------
    float in [0, 1]. 0.0 for disjoint or edge-touching boxes, 1.0 for identical ones.
    """
    # TODO: overlap width, overlap height (both clamped at 0), then inter / (area_a + area_b - inter)
    raise NotImplementedError

In [ ]:
check_iou(my_iou)

In [ ]:
# ---- The real library call ---------------------------------------------------
# torchvision has it batched: box_iou(boxes1, boxes2) -> (N, M) matrix of every pair.
import torch, numpy as np
from torchvision.ops import box_iou
a = torch.tensor([[0., 0., 10., 10.]]); b = torch.tensor([[5., 5., 15., 15.]])
print("torchvision box_iou:", float(box_iou(a, b)[0, 0]))
print("yours              :", my_iou((0, 0, 10, 10), (5, 5, 15, 15)))
assert abs(float(box_iou(a, b)[0, 0]) - my_iou((0,0,10,10), (5,5,15,15))) < 1e-6

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Overlap on the x axis is `min(a.x2, b.x2) - max(a.x1, b.x1)`, and it is only real
if positive - so wrap it in `max(0, ...)`. Same on y.

Union area is `area_a + area_b - intersection`, never the sum. Guard the division
in case both boxes are degenerate and the union is 0.

</details>

In [ ]:
# Sanity picture: watch IoU fall as the second box slides away.
panels = []
a = (20, 30, 120, 130)
for dx in [0, 25, 60, 110]:
    b = (20 + dx, 30, 120 + dx, 130)
    p = np.full((170, 260, 3), 30, np.uint8)
    cv2.rectangle(p, a[:2], a[2:], (0, 255, 0), 2)
    cv2.rectangle(p, b[:2], b[2:], (80, 140, 255), 2)
    cv2.putText(p, f"IoU = {my_iou(a, b):.3f}", (20, 160),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    panels.append(p)
_imgs, _titles = list(panels), [f"shift {d}px" for d in [0, 25, 60, 110]]
fig, axes = plt.subplots(1, len(_imgs), figsize=(3.2 * len(_imgs), 3.2), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

## Task 3 - non-maximum suppression

A detector proposes many overlapping boxes for the same object. NMS is the greedy cleanup:

1. take the highest-scoring box, keep it
2. throw away every remaining box whose IoU with it exceeds the threshold
3. repeat with what is left

Return the **indices** of the kept boxes, highest score first.

In [ ]:
def my_NMSBoxes(boxes, scores, iou_thr):
    """Greedy non-maximum suppression.

    Parameters
    ----------
    boxes   : list of (x1, y1, x2, y2)
    scores  : list of floats, same length
    iou_thr : float. A box is suppressed when its IoU with an already-kept box EXCEEDS this.

    Returns
    -------
    list of kept indices into `boxes`, ordered by descending score.
    Must not modify the input lists. Empty input -> empty list.
    """
    # TODO: sort indices by score descending, then the greedy keep/suppress loop
    raise NotImplementedError

In [ ]:
check_nms(my_NMSBoxes)

In [ ]:
# ---- The real OpenCV / torchvision calls ------------------------------------
# OpenCV: cv2.dnn.NMSBoxes(bboxes, scores, score_threshold, nms_threshold) -> indices
#         CAREFUL: it wants boxes as [x, y, w, h], NOT [x1, y1, x2, y2].
# torch  : torchvision.ops.nms(boxes_xyxy, scores, iou_threshold) -> indices
import cv2, torch, numpy as np
bx = [(0,0,10,10), (1,1,11,11), (50,50,60,60)]
sc = [0.9, 0.8, 0.7]
xywh = [[x1, y1, x2-x1, y2-y1] for x1, y1, x2, y2 in bx]
print("cv2.dnn.NMSBoxes :", np.array(cv2.dnn.NMSBoxes(xywh, sc, 0.05, 0.5)).ravel().tolist())
print("torchvision.nms  :", torch.ops.torchvision.nms(
    torch.tensor(bx, dtype=torch.float), torch.tensor(sc), 0.5).tolist())
print("yours            :", my_NMSBoxes(bx, sc, 0.5))

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

`order = sorted(range(len(boxes)), key=lambda i: -scores[i])` gives you the sort
without touching the caller's lists.

Then: pop the front, append it to `keep`, and rebuild `order` as a list comprehension
filtering out everything with `my_iou(...) > iou_thr`. The whole loop is 4 lines.

Careful with `>` vs `>=`: the grader's chained-boxes case depends on it.

</details>

In [ ]:
# Compare yours against the real thing on synthetic duplicates.
b = [(0,0,10,10), (1,1,11,11), (2,0,12,10), (50,50,60,60)]
s = [0.9, 0.85, 0.8, 0.7]
print("yours:", my_NMSBoxes(b, s, 0.5))
print("torch:", torch.ops.torchvision.nms(torch.tensor(b, dtype=torch.float),
                                          torch.tensor(s), 0.5).tolist())

### NMS on real detections

Ask the model for raw, barely-filtered output, then clean it up with *your* NMS and compare
against the model's own. The counts should be close; where they differ, look at the threshold.

In [ ]:
raw = model("images/messi5.jpg", device=DEVICE, conf=0.10, iou=0.99, verbose=False)[0]
rb = [tuple(x.xyxy[0].tolist()) for x in raw.boxes]
rs = [float(x.conf) for x in raw.boxes]
rl = [model.names[int(x.cls)] for x in raw.boxes]
print(f"raw proposals: {len(rb)}")

keep = my_NMSBoxes(rb, rs, 0.45)
print(f"after your NMS: {len(keep)}")
_imgs, _titles = [my_draw_boxes(rgb, rb, rl, rs, (255, 0, 0)), my_draw_boxes(rgb, [rb[i] for i in keep], [rl[i] for i in keep], [rs[i] for i in keep])], [f"{len(rb)} raw proposals", f"{len(keep)} after your NMS"]
fig, axes = plt.subplots(1, 2, figsize=(6 * 2, 6), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

## What this machine can and cannot do

Measure it rather than trusting a forum post.

In [ ]:
import itertools
imgs = ["images/messi5.jpg", "images/fruits.jpg", "images/board.jpg", "images/building.jpg"]
for dev in ["cpu", DEVICE] if DEVICE != "cpu" else ["cpu"]:
    model.predict(imgs[0], device=dev, verbose=False)          # warm up
    t0 = time.perf_counter()
    for p in itertools.chain(imgs, imgs, imgs):
        model.predict(p, device=dev, verbose=False)
    ms = (time.perf_counter() - t0) / 12 * 1000
    print(f"yolov8n on {dev:4}: {ms:6.1f} ms/image  ({1000/ms:5.1f} fps)")

### Live webcam detection

`cv2.imshow` does not behave inside a notebook on macOS. Write the loop to a file and run it
from a terminal with the 3.12 interpreter. macOS will ask for camera permission the first time.

```bash
cd "/Users/manankapoor/Desktop/SEM5/Image-Processing /CV-Lab"
/Library/Frameworks/Python.framework/Versions/3.12/bin/python3 webcam_yolo.py
```

In [ ]:
script = '''
import cv2, torch
from ultralytics import YOLO

dev = "mps" if torch.backends.mps.is_available() else "cpu"
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise SystemExit("no camera. check System Settings > Privacy & Security > Camera")

while True:
    ok, frame = cap.read()
    if not ok:
        break
    r = model(frame, device=dev, imgsz=480, verbose=False)[0]
    cv2.imshow("yolo - press q to quit", r.plot())
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break
cap.release(); cv2.destroyAllWindows()
'''
open("webcam_yolo.py", "w").write(script)
print("wrote webcam_yolo.py -- run it from a terminal, not from here")

---
## Think about it

1. Raise `conf` from 0.10 to 0.50 on `messi5.jpg`. Which detections vanish? Are they wrong,
   or right-but-uncertain? This is the precision/recall tradeoff with your own eyes on it.
2. NMS is class-agnostic as you wrote it. A dog standing in front of a car produces two
   heavily-overlapping boxes of *different* classes. What does your NMS do, and what should
   a real implementation do? (Look up how torchvision's `batched_nms` handles it.)
3. Your `my_iou` returns 0 for edge-touching boxes. Why is that the correct convention rather
   than a rounding accident?
4. Run detection on `images/baboon.jpg`. Explain the result in terms of the 80 COCO classes.
5. MPS was ~7x faster than CPU on the resnet18 benchmark in notebook 00, but the YOLO gap
   above is much smaller. What is eating the difference at this model size?

---
## What to try next, in order

| next step | why it fits this machine |
|---|---|
| `yolov8n-seg.pt` | same speed class, gives you pixel masks instead of boxes - a genuinely different output |
| **fine-tune** `yolov8n` on a small custom dataset (200-500 images, `imgsz=416`, `batch=4`) | fits in 8 GB unified memory; expect tens of minutes per run, not hours |
| classical feature matching: SIFT + `BFMatcher` + `findHomography` | pure OpenCV, instant on CPU, and it teaches what detection was before CNNs |
| optical flow (`cv2.calcOpticalFlowFarneback`) on webcam frames | real-time on CPU, and motion is the topic image processing alone cannot reach |
| `cv2.dnn` with an ONNX export of your model | shows you the deployment path where torch is not installed |

### Sizing rules for 8 GB unified memory

- **inference:** anything up to `yolov8m` runs fine. `yolov8n`/`s` for interactive work.
- **fine-tuning:** `yolov8n` at `imgsz=416`, `batch=4-8`. Bigger batches will swap and get slower,
  not fail loudly - watch Activity Monitor's memory pressure rather than trusting it to error.
- **from-scratch training** of anything real: not on this machine. Use Colab's free T4 for that,
  and keep this machine for writing the code and iterating on 50-image subsets.
- **watch out:** MPS has no `torch.cuda.amp` equivalent that helps much, and a few ops silently
  fall back to CPU. `PYTORCH_ENABLE_MPS_FALLBACK=1` turns a crash into a slowdown.

---
## API card - drawing and detection

```python
cv2.rectangle(img, pt1, pt2, color, thickness=1, lineType=LINE_8)
cv2.circle(img, center, radius, color, thickness)
cv2.line(img, pt1, pt2, color, thickness)
cv2.putText(img, text, org, fontFace, fontScale, color, thickness)
cv2.polylines(img, [pts], isClosed, color, thickness)
cv2.drawMarker(img, position, color, markerType)
```
All of these **draw in place and return None** — the array you pass in is modified.
Points are `(x, y)` = `(col, row)`, the reverse of numpy indexing. `thickness=-1` fills.
`fontFace` is usually `cv2.FONT_HERSHEY_SIMPLEX`; `org` is the text **baseline**, bottom-left.

```python
indices = cv2.dnn.NMSBoxes(bboxes, scores, score_threshold, nms_threshold)
```
Boxes here are `[x, y, w, h]`, **not** `[x1, y1, x2, y2]`. Mixing this up silently produces
garbage rather than an error.

```python
net = cv2.dnn.readNet(model, config)      # ONNX / Caffe / TF / Darknet
blob = cv2.dnn.blobFromImage(img, scalefactor, size, mean, swapRB, crop)
net.setInput(blob); out = net.forward()
```
`cv2.dnn` is the deployment path when torch is not installed — notebook 18 uses it.